# Tutorial 2: Graph Neural Network for modeling stream-subhalo interactions
Author: Tri Nguyen

## Overview

In this tutorial, we will create a graph neural network (GNN) model to infer dark matter subhalo parameters from stellar stream data.

### Physics Background 
*Stellar streams* are tidal remnants of dwarf galaxies and globular clusters orbiting larger galaxies, such as the Milky Way. 
As stellar streams orbit the host galaxy, they can be perturbed by encounters with dark matter subhalos. 
These perturbations leave imprints on the stream's structure, such as gaps, spurs, and density variations, which can be analyzed to infer properties of the dark matter subhalos.

### Why Graph Neural Networks?
Stellar streams are point clouds in space, where the spatial relationships between particles are crucial for understanding their dynamics and interactions.
GNNs are well-suited to this data structure because they can effectively model relationships and interactions between particles by:
- Treating each particle as a graph node  
- Creating edges between nearby particles (k-nearest neighbors)
- Using attention mechanisms to weight particle contributions

## Installation Requirements

Before running this notebook, ensure you have the following packages installed:

```bash
# Install PyTorch and NumPy (NumPy <2 required for compatibility)
pip install torch torchvision 'numpy<2'

# Install PyTorch Geometric and dependencies
pip install torch-geometric
pip install pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__)")+cpu.html

# Additional dependencies
pip install matplotlib
```

For GPU support, replace `cpu` with your CUDA version (e.g., `cu121`). Visit [PyTorch Geometric Installation](https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html) for more details.

**Note:** PyTorch Geometric installation can sometimes be tricky due to version conflicts. If you encounter issues, try installing the CPU-only version first.

In [ ]:
# Import libraries
import pickle
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

## 0. Download the Data

First, we need to download the simulation data (~500 MB). Run the cell below to download and extract the data automatically.

In [ ]:
import os
import urllib.request
import tarfile

# Create data directory if it doesn't exist
os.makedirs('data', exist_ok=True)

# Check if data already exists
if os.path.exists('data/data_small.pkl'):
    print("Data already exists. Skipping download.")
else:
    print("Downloading data (~500 MB)...")
    
    # Dropbox URL for direct download
    url = "https://www.dropbox.com/scl/fi/gq7jr8dbsrmyflpbdbnj0/data.tar.gz?rlkey=0bdjk230ef5nylg26gg9tl4nh&st=ugsmk6wt&dl=1"
    filename = "data.tar.gz"
    
    # Download the file
    urllib.request.urlretrieve(url, filename)
    print("Download complete. Extracting...")
    
    # Extract the tar.gz file
    with tarfile.open(filename, 'r:gz') as tar:
        tar.extractall('.')
    
    # Clean up the archive
    os.remove(filename)
    print("Data extraction complete!")

## 1. Loading the Simulation Data

We will now load the stream data from the `data` directory. If you haven't already, make sure to run the download cell above. The data is roughly 500 MB in size.

The data contains 10,000 simulated stellar streams, each with a perturbing subhalo with different parameters. The streams are generated using the `StreamSculptor` code from [Nibauer et al. (2024)](https://arxiv.org/abs/2410.21174v1).

**Data Structure:**
- `params_dict`: A dictionary containing the simulation parameters for each stream:
  - `M_sh`: Subhalo mass in solar masses ($\mathrm{M_\odot}$). Shape: `(10000,)`
  - `vz_impact`: Impact velocity in the $z$ direction (line-of-sight) in km/s. Shape: `(10000,)`
- `streams`: The 6D phase-space coordinates of stream stars: $(x, y, z, v_x, v_y, v_z)$
  - Shape: `(10000, N_stars, 6)` where `N_stars = 10000` for all streams in this dataset


In [ ]:
with open('data/data_small.pkl', 'rb') as f:
    data = pickle.load(f)
params_dict = data['params_dict']
streams = data['streams']  # shape: (N_samples, N_particles, 6) # 6 corresponds to (x, y, z, vx, vy, vz)

# params_dict contain two columns: M_sh (the subhalo mass) and vz_impact (the impact velocity along the line of sight)
M_sh = params_dict['M_sh']
vz_impact = params_dict['vz_impact']

We first plot a few example streams for visualization. We will plot 3 streams based on the subhalo mass. 

**More physics context**: 

The subhalo mass affects the strength of the perturbation on the stream. Higher mass subhalos create more pronounced features in the stream, such as larger gaps and more significant density variations. It is degenerate with the impact velocity, as a *slower*-moving subhalo can have a similar effect as a more massive one.


In [ ]:
# Plot a few example streams to visualize the data distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# pick a low, medium, and high mass interacting subhalo
index_Msort = np.argsort(M_sh)
example_indices = [index_Msort[0], index_Msort[5000], index_Msort[-1]]

for i, ax in enumerate(axes):
    ax.plot(
        streams[example_indices[i], :, 1], streams[example_indices[i], :, 2],
        'k.', markersize=1, alpha=0.5)
    ax.set_title(f'Stream Example {i+1}', fontsize=16)
    ax.set_xlabel('y [kpc]', fontsize=16)
    ax.set_ylabel('z [kpc]', fontsize=16)

    # print some info about the stream
    text = f'M_sh: {M_sh[example_indices[i]]:.2e} Msun\nvz_impact: {vz_impact[example_indices[i]]:.2f} km/s'
    ax.text(0.95, 0.05, text, transform=ax.transAxes, fontsize=14,
            va='bottom', ha='right')

plt.tight_layout()
plt.show()

## 2. Preparing the data for the GNN

Unlike Tutorial 1, we are not given a graph structure for the data, so we will first need to convert the 6D phase-space data into graph structure.For simplicity, we will use a $k$-nearest neighbor (kNN) graph, where each node is connected to its $k$-nearest neighbors. Another option for graph construction is a radius graph, where each node is connected to all nodes within a certain radius. 

For the full list of graph transformations, refer to: https://pytorch-geometric.readthedocs.io/en/latest/modules/transforms.html#graph-transforms

In [ ]:
import torch_geometric.transforms as T
from torch_geometric.nn import knn_graph
from torch_geometric.data import Data, Batch

# PyG DataLoader is specialized for loading graphs in batches
from torch_geometric.loader import DataLoader as PyGDataLoader

In [ ]:
# First we will subsample the streams
# To demonstrate the flexibility of the GNN approach, we will sample a random number of particles
# for each stream

def subsample_streams(streams, N_min=90, N_max=110):
    """Subsample the streams to a fixed number of particles"""
    N_samples, N_particles, N_features = streams.shape


    streams_subsampled = []
    for i in range(N_samples):
        N_subsampled = np.random.randint(N_min, N_max + 1)
        indices = np.random.choice(N_particles, N_subsampled, replace=False)
        streams_subsampled.append(streams[i, indices, :])

    return streams_subsampled

# Subsample and prepare data
streams_subsampled = subsample_streams(streams, N_min=90, N_max=110)

In [ ]:
def prepare_training_graph(
    streams, params_dict, train_split=[0.8, 0.1, 0.1],
    batch_size=32, k_neighbors=10):
    """
    Prepare training data for Graph NPE model.
    Constructs graphs once during preprocessing rather than in training loop.

    Args:
        streams: List of arrays, each of shape (N_particles_i, N_features)
            where N_particles_i can vary for each stream.
        params_dict: Dictionary with keys 'M_sh' and 'vz_impact'
        train_split: List of fractions for train/val/test split
        batch_size: Batch size for DataLoader
        k_neighbors: Number of neighbors for k-NN graph construction

    Returns:
        train_loader: PyG DataLoader for training
        val_loader: PyG DataLoader for validation
        norm_dict: Normalization parameters
    """
    N_samples = len(streams)
    N_features = streams[0].shape[1]  # all streams have same feature dimension

    # Prepare parameters (log transform mass)
    params = np.vstack((np.log10(params_dict['M_sh']), params_dict['vz_impact'])).T

    # Split into train/val/test
    # NOTE: the dataset is already shuffled, but in practice you should shuffle before splitting
    N_train = int(train_split[0] * N_samples)
    N_val = int(train_split[1] * N_samples)
    N_test = N_samples - N_train - N_val

    print(f'Number of training samples: {N_train}')
    print(f'Number of validation samples: {N_val}')
    print(f'Number of test samples: {N_test}')

    # Compute normalization statistics on training data

    X_train = np.concat([streams[i] for i in range(N_train)])
    X_train_flat = X_train.reshape(-1, N_features)
    X_loc = torch.tensor(X_train_flat.mean(0), dtype=torch.float32)
    X_scale = torch.tensor(X_train_flat.std(0), dtype=torch.float32)

    y_train_tensor = torch.tensor(params[:N_train], dtype=torch.float32)
    y_min, y_max = y_train_tensor.min(0)[0], y_train_tensor.max(0)[0]
    y_loc = (y_min + y_max) / 2
    y_scale = (y_max - y_min) / 2

    # Create graph transformation
    # set `loop=True` to include self-loops
    transform = T.KNNGraph(k=k_neighbors, loop=True)

    # Create graphs for training data
    print("Constructing graphs for training data...")
    train_graphs = []
    val_graphs = []
    test_graphs = []

    for i in range(N_samples):
        stream, param = streams[i], params[i]

        # Get particle data
        particle_features = torch.tensor(stream, dtype=torch.float32)  # (N_particles, 6)
        pos = particle_features[:, :3]  # (N_particles, 3) - positions for k-NN

        # Normalize features using training statistics
        particle_features_norm = (particle_features - X_loc) / X_scale

        # Normalize parameters
        y_norm = (torch.tensor(param, dtype=torch.float32) - y_loc) / y_scale
        y_norm = y_norm.view(1, -1)  # (1, 2)

        # Create PyG Data object
        data = Data(
            x=particle_features_norm,
            y=y_norm,
            pos=pos
        )
        data = transform(data)

        # Alternatively, we can create and pass `edge_index` directly to `Data`
        # edge_index = knn_graph(pos, k=k_neighbors, loop=True)
        # data = Data(
        #     x=particle_features_norm,
        #     edge_index=edge_index,
        #     y=y_norm,
        #     pos=pos
        # )

        if i < N_train:
            train_graphs.append(data)
        elif i < N_train + N_val:
            val_graphs.append(data)
        else:
            test_graphs.append(data)

    # Create DataLoaders
    train_loader = PyGDataLoader(train_graphs, batch_size=batch_size, shuffle=True)
    val_loader = PyGDataLoader(val_graphs, batch_size=batch_size, shuffle=False)
    test_loader = PyGDataLoader(test_graphs, batch_size=batch_size, shuffle=False)

    norm_dict = {'X_loc': X_loc, 'X_scale': X_scale, 'y_loc': y_loc, 'y_scale': y_scale}

    print("Graph construction complete!\n")
    return train_loader, val_loader, test_loader, norm_dict


train_loader, val_loader, test_loader, norm_dict = prepare_training_graph(
    streams_subsampled, params_dict, train_split=[0.8, 0.1, 0.1],
    batch_size=32, k_neighbors=5
)

In [ ]:
# Take an example batch from the validation loader and plot
# first we will print out the batch to see its data structure
example_batch = next(iter(val_loader))
print(example_batch)

The example batch contains the following data:
- `x`: Node features (6D phase-space coordinates). Shape: `(N_stars, 6)`. 
This is the total number of stars in the batch (i.e. all stars from all streams in the batch concatenated together).
- `edge_index`: Graph connectivity information. Shape: `(2, N_edges)`.
- `y`: Properties of the subhalos. Shape: `(N_batch, 2)`.
- `ptr`: Batch pointer indicating the start index of each stream in the batch. Shape: `(N_batch + 1,)`.
Node features of the $i$-th stream in the batch is stored at `x[ptr[i]:ptr[i+1]]`.
- `batch`: Batch vector assigning each star to its corresponding stream. Shape: `(N_stars,)`.
Node features of the $i$-th stream in the batch can also be accessed using `x[batch == i]`.

In [ ]:
# Plot example graph
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# feel free to change the index to visualize different graphs
example_graph = example_batch[0]

# get node features and edges
pos = example_graph.x[..., :3].numpy()
vel = example_graph.x[..., 3:].numpy()

# plot positions
axes[0, 0].scatter(pos[:, 0], pos[:, 1], c='k', s=1, alpha=1)
axes[0, 1].scatter(pos[:, 1], pos[:, 2], c='k', s=1, alpha=1)
axes[0, 2].scatter(pos[:, 0], pos[:, 2], c='k', s=1, alpha=1)

# plot velocities
axes[1, 0].scatter(vel[:, 0], vel[:, 1], c='k', s=1, alpha=1)
axes[1, 1].scatter(vel[:, 1], vel[:, 2], c='k', s=1, alpha=1)
axes[1, 2].scatter(vel[:, 0], vel[:, 2], c='k', s=1, alpha=1)

# plot edges
for i in range(example_graph.edge_index.shape[1]):
    src = example_graph.edge_index[0, i].item()
    dst = example_graph.edge_index[1, i].item()

    # position
    axes[0, 0].plot(
        [pos[src, 0], pos[dst, 0]], [pos[src, 1], pos[dst, 1]],
        'C0-', alpha=0.2, zorder=-1)
    axes[0, 1].plot(
        [pos[src, 1], pos[dst, 1]], [pos[src, 2], pos[dst, 2]],
        'C0-', alpha=0.2, zorder=-1)
    axes[0, 2].plot(
        [pos[src, 0], pos[dst, 0]], [pos[src, 2], pos[dst, 2]],
        'C0-', alpha=0.2, zorder=-1)

    # velocity
    axes[1, 0].plot(
        [vel[src, 0], vel[dst, 0]], [vel[src, 1], vel[dst, 1]],
        'C1-', alpha=0.2, zorder=-1)
    axes[1, 1].plot(
        [vel[src, 1], vel[dst, 1]], [vel[src, 2], vel[dst, 2]],
        'C1-', alpha=0.2, zorder=-1)
    axes[1, 2].plot(
        [vel[src, 0], vel[dst, 0]], [vel[src, 2], vel[dst, 2]],
        'C1-', alpha=0.2, zorder=-1)

axes[0, 0].set_xlabel('x [normalized units]', fontsize=14)
axes[0, 0].set_ylabel('y [normalized units]', fontsize=14)
axes[0, 1].set_xlabel('y [normalized units]', fontsize=14)
axes[0, 1].set_ylabel('z [normalized units]', fontsize=14)
axes[0, 2].set_xlabel('x [normalized units]', fontsize=14)
axes[0, 2].set_ylabel('z [normalized units]', fontsize=14)
axes[1, 0].set_xlabel('vx [normalized units]', fontsize=14)
axes[1, 0].set_ylabel('vy [normalized units]', fontsize=14)
axes[1, 1].set_xlabel('vy [normalized units]', fontsize=14)
axes[1, 1].set_ylabel('vz [normalized units]', fontsize=14)
axes[1, 2].set_xlabel('vx [normalized units]', fontsize=14)
axes[1, 2].set_ylabel('vz [normalized units]', fontsize=14)

plt.tight_layout()

### 2. Create and train the GNN model

Now we'll create our GNN  model and train it. 
We will use a Graph Attention Network (GAT) architecture for this task ([Velickovic et al. 2018](https://arxiv.org/abs/1710.10903)). 
GAT uses an attention mechanism to weigh the importance of neighboring nodes when aggregating information, allowing the model to focus on the most relevant particles in the stream.

The message passing operation in GAT can be summarized as follows:
$$
h_i^{(l+1)} = \sigma\left( \sum_{j \in \mathcal{N}(i)} \alpha_{ij} W h_j^{(l)} \right)
$$
where:
- $h_i^{(l)}$ is the feature of node $i$ at layer $l$
- $\mathcal{N}(i)$ is the set of neighboring nodes of node $i$
- $\alpha_{ij}$ is the attention coefficient between nodes $i$ and $j$
- $W$ is a learnable weight matrix
- $\sigma$ is a non-linear activation function (e.g., ReLU)

GAT does not require explicit edge features, as the attention mechanism learns to weigh the importance of neighboring nodes based on their features alone.
The attention coefficients $\alpha_{ij}$ are computed as:
$$
\alpha_{ij} = \frac{\exp\left(\text{LeakyReLU}\left(a^T [W h_i^{(l)} || W h_j^{(l)}]\right)\right)}{\sum_{k \in \mathcal{N}(i)} \exp\left(\text{LeakyReLU}\left(a^T [W h_i^{(l)} || W h_k^{(l)}]\right)\right)}
$$
where $a$ is a learnable weight vector and $||$ denotes concatenation.

After passing the node features through several GAT layers, we will use a global pooling layer to aggregate the node features into a single graph-level representation.
Finally, we will use fully connected layers to map the graph representation to the desired output: subhalo mass and impact velocity.

Feel free to modify the architecture and hyperparameters. 
- For the full list of available layers: https://pytorch-geometric.readthedocs.io/en/latest/modules/nn.html#graph-neural-network-layers
- You may also find this cheat sheet helpful: https://pytorch-geometric.readthedocs.io/en/latest/cheatsheet/gnn_cheatsheet.html


**NOTE**: The weight matrix $W$ for computing the attention coefficient is the *same* as the one used to transform the node features in the message passing step. This is a design choice in GAT. Other attention-based GNNs may use separate weight matrices for these two steps.

In [ ]:
import torch
import torch.nn as nn
from torch_geometric.nn import GATConv, global_mean_pool

class GraphModel(nn.Module):
    """
    Graph model using Graph Attention Network (GAT) layers.
    """

    def __init__(self, input_dim, output_dim, hidden_size=64,
                 num_layers=3, heads=4):
        """
        Args:
            input_dim: Input dimension of node features
            output_dim: Output dimension
            hidden_size: Hidden dimension size for GAT layers
            num_layers: Number of GAT layers
            heads: Number of attention heads in GAT
        """
        super().__init__()

        self.input_dim = input_dim
        self.output_dim = output_dim
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.heads = heads

        # Initial projection
        self.input_proj = nn.Linear(input_dim, hidden_size)

        # GAT layers
        self.gat_layers = nn.ModuleList()
        for i in range(num_layers):
            self.gat_layers.append(
                GATConv(hidden_size, hidden_size // heads, heads=heads, concat=True)
            )

        # Final projection to embedding
        self.output_proj = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_dim)
        )

    def forward(self, batch):
        """
        Embed graph using GAT layers and global pooling.

        Args:
            batch: PyG Batch object with pre-constructed graphs

        Returns:
            embeddings: (batch_size, embedding_size)
        """
        x = batch.x
        edge_index = batch.edge_index

        # Initial projection
        x = self.input_proj(x)
        x = torch.relu(x)

        # Apply GAT layers
        for gat_layer in self.gat_layers:
            x = gat_layer(x, edge_index)
            x = torch.relu(x)

        # Global pooling to get graph-level embedding
        x_pooled = global_mean_pool(x, batch.batch)

        # Final projection
        output = self.output_proj(x_pooled)

        return output

model = GraphModel(input_dim=6, output_dim=2, hidden_size=64, num_layers=3, heads=4)
print(model)

# briefly pass the example batch through the model to check everything works
example_shape = model(example_batch).shape  # should be (batch_size, 2)
assert example_shape == (example_batch.num_graphs, 2), \
    f"Output shape mismatch! {example_shape} vs {(example_batch.num_graphs, 2)}"

In [ ]:
def train_model(model, train_loader, val_loader, epochs=50, lr=1e-3, device='cpu'):
    """
    Train the GNN given training and validation data loaders using MSE loss.
    Works with PyG DataLoader that returns (batch_data, batch_params).
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    device = torch.device(device)
    model.to(device)

    criterion = nn.MSELoss()

    train_losses = []
    val_losses = []
    print(f"Training for {epochs} epochs...\n")

    for epoch in range(epochs):

        model.train()
        epoch_loss = 0.0
        n_batches = 0
        for batch_data in train_loader:
            optimizer.zero_grad()
            y_true = batch_data.y.to(device)
            y_pred = model(batch_data.to(device))
            loss = criterion(y_pred, y_true)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        avg_loss = epoch_loss / n_batches
        train_losses.append(avg_loss)

        # compute the validation loss at the end of each epoch
        model.eval()
        val_epoch_loss = 0.0
        n_val_batches = 0
        with torch.no_grad():
            for batch_data in val_loader:
                y_true = batch_data.y.to(device)
                y_pred = model(batch_data.to(device))
                val_loss = criterion(y_pred, y_true)

                val_epoch_loss += val_loss.item()
                n_val_batches += 1
        val_avg_loss = val_epoch_loss / n_val_batches
        val_losses.append(val_avg_loss)

        # print progress every epoch
        print(f"Epoch {epoch+1:3d}/{epochs} | Train Loss: {avg_loss:.4f} | Val Loss: {val_avg_loss:.4f}")

    print("\nTraining complete!")
    return train_losses, val_losses

In [ ]:
train_losses, val_losses = train_model(
    model, train_loader, val_loader,
    epochs=50,  # reduce epochs for quicker testing, but I've found that this model requires at least 50 epochs to converge
    lr=1e-4,  # learning rate, in my experience, lower lr works better for GNNs
    device='cpu'  # change to cuda to use GPU if available
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(train_losses, label='Train Loss')
ax.plot(val_losses, label='Validation Loss')
ax.set_xlabel('Epoch', fontsize=16)
ax.set_ylabel('MSE Loss', fontsize=16)
ax.legend(fontsize=14)
plt.show()

### 3. Validation Test

Now, we will validate the performance of our trained GNN model on the hold-out test set.

In [ ]:
from sklearn.metrics import r2_score

In [ ]:
# evaluate on test set
test_predictions = []
test_true = []
model.eval()
with torch.no_grad():
    for batch_data in test_loader:
        y_true = batch_data.y
        y_pred = model(batch_data.to('cpu'))
        test_predictions.append(y_pred.cpu().numpy())
        test_true.append(y_true.cpu().numpy())
test_predictions = np.vstack(test_predictions)
test_true = np.vstack(test_true)

# unnormalize predictions and true values
y_loc = norm_dict['y_loc'].numpy()
y_scale = norm_dict['y_scale'].numpy()
test_predictions_unnorm = test_predictions * y_scale + y_loc
test_true_unnorm = test_true * y_scale + y_loc

In [ ]:
# calculate R2 score while we're at it
r2_Msh = r2_score(test_true_unnorm[:, 0], test_predictions_unnorm[:, 0])
r2_vz = r2_score(test_true_unnorm[:, 1], test_predictions_unnorm[:, 1])

print(f"Test R2 Score for log10(M_sh): {r2_Msh:.4f}")
print(f"Test R2 Score for v_z: {r2_vz:.4f}")

# plot true vs predicted
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# True vs Predicted for log10(M_sh)
axes[0].scatter(
    test_true_unnorm[:, 0], test_predictions_unnorm[:, 0],
    c='k', s=5, alpha=0.5, zorder=2)
axes[0].plot([6, 8], [6, 8], 'k--', lw=2, zorder=1)
axes[0].set_xlim(6, 8)
axes[0].set_ylim(6, 8)
axes[0].set_xlabel(r'True $\log_{10}(M_{sh})$', fontsize=16)
axes[0].set_ylabel(r'Predicted $\log_{10}(M_{sh})$', fontsize=16)

# True vs Predicted for v_z
axes[1].scatter(
    test_true_unnorm[:, 1], test_predictions_unnorm[:, 1],
    c='k', s=5, alpha=0.5, zorder=2)
axes[1].plot([-50, 50], [-50, 50], 'k--', lw=2, zorder=1)
axes[1].set_xlim(-50, 50)
axes[1].set_ylim(-50, 50)
axes[1].set_xlabel(r'True $v_z$ [km/s]', fontsize=16)
axes[1].set_ylabel(r'Predicted $v_z$ [km/s]', fontsize=16)

plt.tight_layout()

We find that the model performs well in predicting subhalo masses and impact velocities from the stellar stream data. 
The model does not give uncertainties, as we only trained on MSE loss.
For scientific applications, techniques such as simulation-based inference or Bayesian neural networks can be used to estimate uncertainties.
GNNs can then be adapted as the "feature extractor" or "embedding network" in these frameworks.

As always, reminder that as with any machine learning model, one should always stress-test the model on out-of-distribution data to ensure robustness :)

# Summary
In this tutorial, we have built a graph neural network model to infer dark matter subhalo parameters from stellar stream data.
We have covered the following steps:
1. Loading and understanding the simulation data
2. Preparing the data for the GNN
3. Building and training the GNN model
4. Evaluating the model's performance